In [1]:
import sys
import warnings

sys.path.append("..")
sys.dont_write_bytecode = True
warnings.filterwarnings("ignore")

In [ ]:
import polars as pl
from utils.utils import (
    split_train_test,
    get_parent_run_id_from_experiment,
    compute_generalisation_error_from_run_id_and_experiment_id,
)
from utils.statics import lightgbm_model_name, FEATUERE_ENGINEERING_EXPERIMENT_ID
from utils.preprocessing_handler import PreprocessingHandler
from feature_engineering.RowWiseTransformations import RowWiseTransformations
from feature_engineering.OpenFETransformations import OpenFETransformations
from feature_engineering.ColumnTransformer import ColumnTransformer
from model.nested_cv_eval import ModelCVEvaluator

model_type = lightgbm_model_name

In [3]:
passes_df = pl.read_parquet("../data/02-analysis/passes.parquet")

In [4]:
train_df, test_df = split_train_test(passes_df=passes_df)

In [5]:
numerical_columns = [
    "start_x",
    "start_y",
    "end_x",
    "end_y",
    "length",
    "angle",
    "duration",
]

columns = train_df.columns
target_column = "outcome"
categorical_columns = [col for col in columns if col not in numerical_columns and col != target_column]

preprocessing_handler = PreprocessingHandler(df=train_df, categorical_columns=categorical_columns)
train_df_temp = preprocessing_handler.preprocess_categorical_columns()
train_df_temp = preprocessing_handler.preprocess_outcome_column()
y_train = train_df_temp.select("outcome").to_series()
X_train = preprocessing_handler.preprocess_outcome_column().drop(pl.col("outcome"))

In [6]:
categorical_columns = ["body_part", "height"]
column_transformer_config = {
            "cat_columns": categorical_columns
        }

In [ ]:
row_wise_transformations = RowWiseTransformations()
column_wise_transformations = ColumnTransformer(**column_transformer_config)
open_fe_transformations = OpenFETransformations(n_features=5)

In [8]:
run_name = f"Feature_engineering_11_OFE"
experiment_name = "Feature Engineering"
model_cv_evaluator = ModelCVEvaluator(
    model_type=model_type,
    row_wise_transformations=row_wise_transformations,
    column_wise_transformations=column_wise_transformations,
    open_fe_transformations=open_fe_transformations,
    n_inner_splits=3,
    n_outer_splits=3,
    n_trials=50,
    use_hyperparameter_tuning=False,
    use_feature_selection=False,
    use_feature_engineering=True,
    use_ofe=True,
    log_feature_importance=True,
    log_parameter_importance=True,
    run_name=run_name,
    experiment_name=experiment_name,
    categorical_columns=categorical_columns,
)

In [9]:
result_cv = model_cv_evaluator.get_generalisation_error(X_train=X_train, y_train=y_train)

2026-04-19 16:38:58,669 - INFO - Starting training with model lightgbm with the following configuration:
        - 3 inner splits
        - 3 outer splits
        - 50 trials
2026-04-19 16:39:00,430 - INFO - Fitting OpenFE on fold 1


The number of candidate features is 1775
Start stage I selection.


100%|██████████| 32/32 [00:19<00:00,  1.66it/s]


486 same features have been deleted.
Meet early-stopping in successive feature-wise halving.


100%|██████████| 32/32 [00:56<00:00,  1.75s/it]


The number of remaining candidate features is 420
Start stage II selection.


100%|██████████| 30/30 [00:23<00:00,  1.29it/s]


Finish data processing.
🏃 View run Outer_fold_1 at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/959422f540fe4432a25d5ba62a9df5c3
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708
🏃 View run Feature_engineering_11_OFE_lightgbm at: http://127.0.0.1:8080/#/experiments/401199845079117708/runs/e225903cea5448c992b2977d84ad1ab3
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/401199845079117708


In [17]:
result_cv[1].children[0].name

'duration'

In [11]:
# parent_run_id = get_parent_run_id_from_experiment(result=result_cv, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID)
# compute_generalisation_error_from_run_id_and_experiment_id(
#     parent_run_id=parent_run_id, experiment_id=FEATUERE_ENGINEERING_EXPERIMENT_ID
# )